# Pseudo-Differential Equation Solvers & Ray Trajectories

This notebook demonstrates numerical simulations for pseudo-differential equations (PDEs), matrix systems, and ray dynamics using the `pde_solver_exponential` package.

## 1. Imports and Setup

Import necessary scientific libraries along with core solvers, visualization utilities, and trajectory integrators.

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from psiop import * 

## 2. Example A: Scalar 1D Variable-Coefficient Advection-Diffusion

Solves the 1D initial value problem with variable wave speed $c(x) = 0.5 + 0.3\sin(x)$ and viscosity $\nu = 0.02$:

$$\partial_t u = -c(x) \partial_x u + \nu \partial_{xx} u$$

The corresponding symbol is $s(x, \xi) = -i c(x) \xi - \nu \xi^2$.

In [ ]:
x, xi = sp.symbols('x xi', real=True)
c = 0.5 + 0.3 * sp.sin(x)
nu = 0.02
s_scalar = -sp.I * c * xi - nu * xi**2
f_gauss = lambda X: np.exp(-X**2)

t, U, (xg, kxg) = solve_first_order(
    s_scalar, [x], f_gauss, dt=0.02, n_steps=200, order=3, L=10.0, N=256,
    apply_kwargs=dict(freq_window='gaussian')
)
plot_scalar_1d(t, U, xg, title="Advection-Diffusion", quantity='real')

## 3. Example B: 2x2 Hyperbolic Matrix System

Evolves a coupled 2D vector system $\mathbf{u}(t, x) = [u_1, u_2]^T$ with non-diagonal symbol matrix:

$$S(x, \xi) = \begin{pmatrix} 0 & i\xi \\ i\xi & 0 \end{pmatrix}$$

In [ ]:
s_matrix = sp.Matrix([[0, sp.I * xi], [sp.I * xi, 0]])
f_vec = lambda X: [np.exp(-X**2), np.zeros_like(X)]

t2, U2, (xg2, kxg2) = solve_first_order(
    s_matrix, [x], f_vec, dt=0.02, n_steps=200, order=4, L=10.0, N=256,
    apply_kwargs=dict(freq_window='gaussian')
)
plot_matrix_1d(t2, U2, xg2, labels=["u1", "u2"], quantity='real')

## 4. Example C: Scalar 1D Pure Transport & Animation

Simulates pure linear advection $\partial_t u + 1.5 \partial_x u = 0$ and creates an animated trajectory plot of the field profile over time.

In [ ]:
c_speed = 1.5
s_transport = -sp.I * c_speed * xi

t3, U3, (xg3, kxg3) = solve_first_order(
    s_transport, [x], f_gauss, dt=0.02, n_steps=200, order=2, L=10.0, N=256,
    apply_kwargs=dict(freq_window='gaussian')
)
anim = animate_scalar_1d(t3, U3, xg3, quantity='real', interval=30)

HTML(anim.to_jshtml())

## 5. Example E: 2D Wavepacket Propagation

Evolves a 2D spatial Gaussian wavepacket under the Hamiltonian operator across a 2D numerical spatial grid.

In [ ]:
x, y, xi, eta = sp.symbols('x y xi eta', real=True)
V_hh = (x**2 + y**2)/2 + x**2*y - y**3/3
H_hh = (xi**2 + eta**2)/2 + V_hh
s_hh = -sp.I * H_hh

def f_wavepacket(X, Y):
    gauss = np.exp(-((X - 0.1)**2)/(2*0.5**2) - ((Y - 0.1)**2)/(2*0.5**2))
    phase = np.exp(1j * (0.45 * X + 0.35 * Y))
    return gauss * phase

t_hh, U_hh, grids_hh = solve_first_order(
    s_hh, [x, y], f_wavepacket, dt=0.005, n_steps=200, order=2, L=6.0, N=96,
    apply_kwargs=dict(freq_window='gaussian')
)
xg_hh, yg_hh, _, _ = grids_hh
plot_scalar_2d(t_hh, U_hh, xg_hh, yg_hh, quantity='abs')